In [2]:
# warnings 안뜨게 하는 코드
import warnings
warnings.filterwarnings('ignore')

### - 손 모션캡쳐 기본 코드

In [1]:
import cv2  # OpenCV 라이브러리 (이미지/비디오 처리)
import mediapipe as mp  # MediaPipe 라이브러리 (손 인식 AI 모델)

# MediaPipe에서 제공하는 그리기 도구 및 스타일 설정 로드
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands  # 손 인식 솔루션 모듈 로드

# 'hand.mp4' 영상을 불러옴 (숫자 0을 넣으면 웹캠 사용)
cap = cv2.VideoCapture('sign_language.mp4')

# 손 인식 모델 설정
with mp_hands.Hands(
    model_complexity=0,      # 모델 복잡도 (0 또는 1, 낮은 수일수록 빠르지만 정확도는 낮음)
    min_detection_confidence=0.5,  # 최소 감지 신뢰도 (0.5초과 시 손으로 판단)
    min_tracking_confidence=0.5) as hands: # 최소 추적 신뢰도 (손을 놓치지 않고 따라가는 기준)

  while cap.isOpened():  # 영상이 열려있는 동안 반복
    success, image = cap.read()  # 프레임별로 영상을 읽음
    if not success:  # 영상이 끝나거나 읽기에 실패하면 종료
      break

    # 성능 향상을 위해 이미지를 읽기 전용으로 설정
    image.flags.writeable = False
    # OpenCV는 BGR 순서로 색상을 읽지만, MediaPipe는 RGB를 사용하므로 색상 공간 변환
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # AI 모델이 이미지에서 손의 랜드마크(마디)를 추출하여 결과 반환
    results = hands.process(image)

    # 이미지를 다시 화면에 보여주기 위해 쓰기 가능 모드로 변경 및 BGR로 재변환
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # 결과값에 손 마디 정보(landmarks)가 있다면 화면에 그리기 시작
    if results.multi_hand_landmarks:
      for hand_landmarks in results.multi_hand_landmarks:
        # 이미지 위에 손 마디 점과 연결선을 그림
        mp_drawing.draw_landmarks(
            image,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS, # 마디끼리의 연결 관계
            mp_drawing_styles.get_default_hand_landmarks_style(), # 기본 점 스타일
            mp_drawing_styles.get_default_hand_connections_style()) # 기본 선 스타일

    # 이미지를 좌우 반전(selfie-view)하여 윈도우 창에 출력
    cv2.imshow('MediaPipe Hands', cv2.flip(image, 1))
    
    # 'q' 키를 누르면 루프 탈출 및 종료
    if cv2.waitKey(5) & 0xFF == ord('q'):
      break

# 자원 해제 및 모든 윈도우 창 닫기
cap.release()
cv2.destroyAllWindows()

C:\Users\User\.conda\envs\mp310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


### - 5개 손가락 끝에 원 그리기, 손가락 번호 달기

In [33]:
import cv2  # OpenCV 라이브러리 (이미지/비디오 처리)
import mediapipe as mp  # MediaPipe 라이브러리 (손 인식 AI 모델)

# MediaPipe에서 제공하는 그리기 도구 및 스타일 설정 로드
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands  # 손 인식 솔루션 모듈 로드

# 'hand.mp4' 영상을 불러옴 (숫자 0을 넣으면 웹캠 사용)
cap = cv2.VideoCapture('hand.mp4')

# 손 인식 모델 설정
with mp_hands.Hands(
    model_complexity=0,      # 모델 복잡도 (0 또는 1, 낮은 수일수록 빠르지만 정확도는 낮음)
    min_detection_confidence=0.5,  # 최소 감지 신뢰도 (0.5초과 시 손으로 판단)
    min_tracking_confidence=0.5) as hands: # 최소 추적 신뢰도 (손을 놓치지 않고 따라가는 기준)

  while cap.isOpened():  # 영상이 열려있는 동안 반복
    success, image = cap.read()  # 프레임별로 영상을 읽음
    if not success:  # 영상이 끝나거나 읽기에 실패하면 종료
      break

    # 성능 향상을 위해 이미지를 읽기 전용으로 설정
    image.flags.writeable = False
    # OpenCV는 BGR 순서로 색상을 읽지만, MediaPipe는 RGB를 사용하므로 색상 공간 변환
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # AI 모델이 이미지에서 손의 랜드마크(마디)를 추출하여 결과 반환
    results = hands.process(image)

    # 이미지를 다시 화면에 보여주기 위해 쓰기 가능 모드로 변경 및 BGR로 재변환
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # 결과값에 손 마디 정보(landmarks)가 있다면 화면에 그리기 시작
    if results.multi_hand_world_landmarks:
        for hand_world_landmarks in results.multi_hand_world_landmarks:
            # 중지 손가락 끝의 실제 미터 단위 추정 좌표
            middle_tip = hand_world_landmarks.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP]
            # 값이 0.05 라면 손목에서 약 5cm 정도 떨어져 있다는 뜻
            # print(f"손목 기준 중지 끝 깊이: {middle_tip.z:.4f}m")
        
        for hand_landmarks in results.multi_hand_landmarks:
            # 8번 마디: 검지 손가락 끝 (INDEX_FINGER_TIP)
            index_finger_tip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]            
            # x, y, z 값 출력
            # print(f"X: {index_finger_tip.x}, Y: {index_finger_tip.y}, Z: {index_finger_tip.z}")
            
            # 이미지 위에 손 마디 점과 연결선을 그림
            mp_drawing.draw_landmarks(
                image,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS, # 마디끼리의 연결 관계
                mp_drawing_styles.get_default_hand_landmarks_style(), # 기본 점 스타일
                mp_drawing_styles.get_default_hand_connections_style()) # 기본 선 스타일

            # 2. 특정 마디(4, 8, 12, 16, 20)에 원 그리기
            target_indices = [4, 8, 12, 16, 20]
            h, w, c = image.shape # 이미지의 세로, 가로 크기 가져오기
    
            for idx in target_indices:
                # 해당 인덱스의 랜드마크 데이터 가져오기
                landmark = hand_landmarks.landmark[idx]
                
                # MediaPipe는 0~1 사이의 상대 좌표를 주므로, 이미지 크기를 곱해 실제 픽셀 좌표로 변환
                cx, cy = int(landmark.x * w), int(landmark.y * h)
                
                # cv2.circle(대상이미지, (중심좌표), 반지름, 색상(BGR), 두께)
                # 여기서는 반지름 10, 노란색(0, 255, 255), 채워진 원(-1)으로 그림
                cv2.circle(image, (cx, cy), 20, (0, 255, 255), -1)
                
                # (선택) 번호도 같이 표시하고 싶다면 아래 주석 해제
                cv2.putText(image, str(idx), (cx, cy - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # 이미지를 좌우 반전(selfie-view)하여 윈도우 창에 출력
    cv2.imshow('MediaPipe Hands', cv2.flip(image, 1))
    
    # 'q' 키를 누르면 루프 탈출 및 종료
    if cv2.waitKey(1) & 0xFF == ord('q'): break

# 자원 해제 및 모든 윈도우 창 닫기
cap.release()
cv2.destroyAllWindows()

### - 8번 손가락 검지로 선그리기

In [32]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# 궤적을 저장할 리스트 (검지 끝 좌표들)
points = []

cap = cv2.VideoCapture('hand.mp4')

with mp_hands.Hands(model_complexity=0, min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        image = cv2.flip(image, 1) # 보기 편하게 좌우 반전
        h, w, c = image.shape
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # 8번 마디(검지 끝) 좌표 가져오기
                index_finger_tip = hand_landmarks.landmark[8]
                cx, cy = int(index_finger_tip.x * w), int(index_finger_tip.y * h)
                
                # 검지 끝 위치 저장
                points.append((cx, cy))
                
                # 현재 손가락 위치에 작은 원 그리기
                cv2.circle(image, (cx, cy), 5, (0, 255, 0), -1)

        # 저장된 모든 점들을 선으로 잇기
        for i in range(1, len(points)):
            if points[i - 1] is None or points[i] is None:
                continue
            # 선 그리기: cv2.line(이미지, 시작점, 끝점, 색상, 두께)
            cv2.line(image, points[i - 1], points[i], (255, 0, 0), 3)

        cv2.imshow('Air Drawing', image)

        # 'q'를 누르면 종료, 'c'를 누르면 그림 지우기
        key = cv2.waitKey(5) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('c'):
            points = []

cap.release()
cv2.destroyAllWindows()

### - 엄지검지 글자쓰기 (실패)

In [41]:
import cv2
import mediapipe as mp
import math

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
points = []

cap = cv2.VideoCapture('hand_text2.mp4')

with mp_hands.Hands(model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        # [회전/반전] 사용자 환경에 맞게 조정된 값 사용
        image = cv2.rotate(image, cv2.ROTATE_180)
        image = cv2.flip(image, -1)

        h, w, c = image.shape
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # 뼈다귀 그리기 (인식 확인용)
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                # 4번(엄지 끝)과 8번(검지 끝) 좌표 추출
                thumb = hand_landmarks.landmark[4]
                index = hand_landmarks.landmark[8]

                # 두 점 사이의 유클리드 거리 계산 (픽셀 단위 변환 후)
                dist = math.sqrt((thumb.x - index.x)**2 + (thumb.y - index.y)**2)

                # 거리 기준 (0.05는 화면 너비의 5% 정도 거리를 의미)
                # 이 값을 조절해서 '클릭' 감도를 맞추세요!
                if dist < 0.1: 
                    cx, cy = int(index.x * w), int(index.y * h)
                    points.append((cx, cy))
                    # 클릭 중임을 표시하는 원 (빨간색)
                    cv2.circle(image, (cx, cy), 10, (0, 0, 255), -1)
                else:
                    if points and points[-1] is not None:
                        points.append(None)

        # 궤적 그리기
        for i in range(1, len(points)):
            if points[i - 1] is None or points[i] is None: continue
            cv2.line(image, points[i - 1], points[i], (255, 0, 0), 5)

        cv2.imshow('Pinch Writing', image)
        if cv2.waitKey(10) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [34]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils # 뼈다귀 그리기용
points = []

cap = cv2.VideoCapture('hand_text.mp4')

with mp_hands.Hands(
    model_complexity=1, # 0보다 정확도가 높음
    min_detection_confidence=0.3, # 인식을 못 하면 이 숫자를 더 낮춰보세요 (0.1까지)
    min_tracking_confidence=0.3) as hands:
    
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        # [수정] 좌우 반전 (거울 모드)
        # flipCode 1은 좌우 반전입니다. 만약 이미 반전된 영상이면 이 줄을 지우세요.
        image = cv2.flip(image, -1)
        # 1. 영상 회전 (필요에 따라 90도씩 회전)
        # cv2.ROTATE_90_CLOCKWISE: 시계방향 90도
        # cv2.ROTATE_180: 180도
        # cv2.ROTATE_90_COUNTERCLOCKWISE: 반시계방향 90도
        image = cv2.rotate(image, cv2.ROTATE_180)

        h, w, c = image.shape
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # [디버깅] AI가 손을 찾고 있는지 뼈다귀를 화면에 그려봅니다.
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                
                index_tip = hand_landmarks.landmark[8]
                
                # [중요] 터미널 창에 찍히는 Z값을 보세요. 
                # 책상에 닿았을 때 숫자가 어떻게 변하는지 확인해야 기준값을 잡습니다.
                # print(f"Z: {index_tip.z:.4f}") 

                # 바닥 인식 기준 (Z값이 작을수록 카메라와 가까운 것)
                # 처음보다 안 그려진다면 기준값이 너무 엄격해서 그럴 수 있습니다.
                # -0.05 정도로 더 넉넉하게 바꿔보세요.
                if index_tip.z < -0.0001: 
                    cx, cy = int(index_tip.x * w), int(index_tip.y * h)
                    points.append((cx, cy))
                else:
                    if points and points[-1] is not None:
                        points.append(None)

        # 궤적 그리기
        for i in range(1, len(points)):
            if points[i - 1] is None or points[i] is None: continue
            cv2.line(image, points[i - 1], points[i], (255, 0, 0), 5)

        cv2.imshow('Debug Mode', image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

### - 가위 바위 보 인식

In [12]:
import cv2
import mediapipe as mp
import os

# 파일이 있는지 확인하는 함수 (디버깅용)
def load_img(path):
    img = cv2.imread(path)
    if img is None:
        print(f"경고: {path} 파일을 찾을 수 없습니다!")
        # 파일이 없으면 빈 검은색 이미지라도 생성해서 에러 방지
        return np.zeros((150, 150, 3), dtype="uint8")
    # 이미지 크기를 150x150으로 통일 (너무 크면 영상 가림)
    return cv2.resize(img, (150, 150))

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# 1. 이미지 로드 (경로를 정확히 맞춰주세요)
img_rock = load_img('rock.png')
img_scissors = load_img('scissors.png')
img_paper = load_img('paper.png')

cap = cv2.VideoCapture('hand3.mp4')

with mp_hands.Hands(model_complexity=1, min_detection_confidence=0.7, min_tracking_confidence=0.7) as hands:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        # 회전 및 반전 (기존 설정 유지)
        image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
        image = cv2.flip(image, 1)
        
        h, w, _ = image.shape
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)

        target_img = None # 화면에 표시할 이미지 저장 변수

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                # 손가락 개수 파악 (접힘/펴짐)
                # 8(검지), 12(중지), 16(약지), 20(소지)
                cnt = 0
                for i in [8, 12, 16, 20]:
                    if hand_landmarks.landmark[i].y < hand_landmarks.landmark[i-2].y:
                        cnt += 1

                # 판정 로직
                if cnt == 4:     # 보
                    target_img = img_paper
                elif cnt == 2:   # 가위
                    target_img = img_scissors
                elif cnt == 0:   # 바위
                    target_img = img_rock

        # [핵심] 왼쪽 상단에 이미지 합성
        if target_img is not None:
            gh, gw, _ = target_img.shape
            # 영상 왼쪽 상단(좌표 20, 20)에 이미지 배치
            # 이미지의 픽셀 값을 영상 픽셀 값에 직접 대입
            image[20:20+gh, 20:20+gw] = target_img
            
            # 어떤 상태인지 글씨도 크게 써주기 (확인용)
            cv2.putText(image, "STATE: DETECTED", (20, 200), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow('Rock Paper Scissors', image)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [3]:
import cv2
import mediapipe as mp
import math
import numpy as np

# 파일이 있는지 확인하는 함수 (디버깅용)
def load_img(path):
    img = cv2.imread(path)
    if img is None:
        print(f"경고: {path} 파일을 찾을 수 없습니다!")
        # 파일이 없으면 빈 검은색 이미지라도 생성해서 에러 방지
        return np.zeros((300, 300, 3), dtype="uint8")
    # 이미지 크기를 300x300으로 통일 (너무 크면 영상 가림)
    return cv2.resize(img, (300, 300))

# [2] 두 점 사이의 거리를 구하는 함수 (속도를 위해 math.sqrt 사용)
def get_dist(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# 1. 이미지 로드 (경로를 정확히 맞춰주세요)
img_rock = load_img('rock.png')
img_scissors = load_img('scissors.png')
img_paper = load_img('paper.png')

cap = cv2.VideoCapture('hand3.mp4')

# 속도를 위해 model_complexity를 0으로 낮추고, 인식 신뢰도를 조절합니다.
with mp_hands.Hands(model_complexity=0, min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
        image = cv2.flip(image, 1)
        h, w, _ = image.shape
        results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        display_img = None
        label = "Unknown" # 기본 라벨

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                # 손가락 끝(Tip)과 손목(Wrist) 사이의 거리 측정
                # 손가락이 펴졌는지 판단하는 기준: 끝마디(Tip)가 중간마디(Pip)보다 손목에서 더 멀어야 함
                fingers = []
                for tip, pip in [(8, 6), (12, 10), (16, 14), (20, 18)]:
                    dist_tip = get_dist(hand_landmarks.landmark[tip], hand_landmarks.landmark[0])
                    dist_pip = get_dist(hand_landmarks.landmark[pip], hand_landmarks.landmark[0])
                    fingers.append(dist_tip > dist_pip)

                # 엄지(4번)는 특성상 옆으로 벌려졌는지를 봅니다 (선택사항)
                thumb_dist = get_dist(hand_landmarks.landmark[4], hand_landmarks.landmark[17]) # 엄지 끝과 새끼 뿌리 거리
                
                up_count = fingers.count(True)

                # 판단 로직 (더 직관적으로 변경)
                if up_count == 4:
                    label = "PAPER"
                    display_img = img_paper
                elif up_count == 2 and fingers[0] and fingers[1]: # 검지, 중지만 펴짐
                    label = "SCISSORS"
                    display_img = img_scissors
                elif up_count <= 1: # 엄지를 제외하고 거의 다 접힘
                    label = "ROCK"
                    display_img = img_rock

        # 이미지 합성 및 글씨 출력
        if display_img is not None:
            image[20:320, 20:320] = display_img
        
        # 글씨가 실시간으로 바뀌는지 확인
        cv2.putText(image, f"RESULT: {label}", (20, 370), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

        cv2.imshow('Fast RPS AI', image)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

### - 카메라 사용(가위바위보)

In [36]:
import cv2
import mediapipe as mp
import math
import numpy as np

# 파일이 있는지 확인하는 함수 (디버깅용)
def load_img(path):
    img = cv2.imread(path)
    if img is None:
        print(f"경고: {path} 파일을 찾을 수 없습니다!")
        # 파일이 없으면 빈 검은색 이미지라도 생성해서 에러 방지
        return np.zeros((300, 300, 3), dtype="uint8")
    # 이미지 크기를 300x300으로 통일 (너무 크면 영상 가림)
    return cv2.resize(img, (300, 300))

# [2] 두 점 사이의 거리를 구하는 함수 (속도를 위해 math.sqrt 사용)
def get_dist(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# 1. 이미지 로드 (경로를 정확히 맞춰주세요)
img_rock = load_img('rock.png')
img_scissors = load_img('scissors.png')
img_paper = load_img('paper.png')

cap = cv2.VideoCapture(0)

# 속도를 위해 model_complexity를 0으로 낮추고, 인식 신뢰도를 조절합니다.
with mp_hands.Hands(model_complexity=0, min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        image = cv2.rotate(image, cv2.ROTATE_180)
        # image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
        image = cv2.flip(image, 0)
        h, w, _ = image.shape
        results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        display_img = None
        label = "Unknown" # 기본 라벨

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                # 손가락 끝(Tip)과 손목(Wrist) 사이의 거리 측정
                # 손가락이 펴졌는지 판단하는 기준: 끝마디(Tip)가 중간마디(Pip)보다 손목에서 더 멀어야 함
                fingers = []
                for tip, pip in [(8, 6), (12, 10), (16, 14), (20, 18)]:
                    dist_tip = get_dist(hand_landmarks.landmark[tip], hand_landmarks.landmark[0])
                    dist_pip = get_dist(hand_landmarks.landmark[pip], hand_landmarks.landmark[0])
                    fingers.append(dist_tip > dist_pip)

                # 엄지(4번)는 특성상 옆으로 벌려졌는지를 봅니다 (선택사항)
                thumb_dist = get_dist(hand_landmarks.landmark[4], hand_landmarks.landmark[17]) # 엄지 끝과 새끼 뿌리 거리
                
                up_count = fingers.count(True)

                # 판단 로직 (더 직관적으로 변경)
                if up_count == 4:
                    label = "PAPER"
                    display_img = img_paper
                elif up_count == 2 and fingers[0] and fingers[1]: # 검지, 중지만 펴짐
                    label = "SCISSORS"
                    display_img = img_scissors
                elif up_count <= 1: # 엄지를 제외하고 거의 다 접힘
                    label = "ROCK"
                    display_img = img_rock

        # 이미지 합성 및 글씨 출력
        if display_img is not None:
            image[20:320, 20:320] = display_img
        
        # 글씨가 실시간으로 바뀌는지 확인
        cv2.putText(image, f"RESULT: {label}", (20, 370), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

        cv2.imshow('Fast RPS AI', image)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [7]:
# !pip install mediapipe opencv-python numpy

### - 카메라 사용(글쓰기)

In [35]:
import cv2
import mediapipe as mp
import numpy as np

# 성능 최적화 설정
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands=1,
    model_complexity=0, # 속도 우선
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

cap = cv2.VideoCapture(0)
canvas = None
xp, yp = 0, 0
SMOOTHING = 0.4 # 글씨 노이즈를 잡기 위해 필터값 조정

print("--- 엄지 스위치 모드 ---")
print("1. 엄지 펼치기: 쓰기 (Green)")
print("2. 엄지 접기: 이동 (Red)")
print("3. 주먹 쥐기: 지우개 (White)")

while cap.isOpened():
    success, img = cap.read()
    if not success: break

    img = cv2.flip(img, -1)
    h, w, c = img.shape
    if canvas is None: canvas = np.zeros_like(img)

    result = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            lm = hand_landmarks.landmark
            
            # 중지 끝(12)을 펜촉 좌표로 사용 (검지보다 움직임이 덜함)
            cx, cy = int(lm[12].x * w), int(lm[12].y * h)
            
            # --- 제스처 인식 로직 ---
            # 1. 엄지 펼침 여부 (4번 끝이 2번 마디보다 바깥에 있는지)
            # 좌우 반전 상태이므로 x좌표 비교 (오른손 기준)
            thumb_open = lm[4].x < lm[2].x 
            
            # 2. 다른 손가락들이 접혔는지 (주먹/지우개 판단용)
            # 검지(8), 중지(12), 약지(16), 소지(20)
            fingers_up = [lm[i].y < lm[i-2].y for i in [8, 12, 16, 20]]

            # --- 상태 결정 ---
            if sum(fingers_up) == 0: # A. 주먹: 지우개
                xp, yp = 0, 0
                cv2.circle(canvas, (cx, cy), 60, (0, 0, 0), cv2.FILLED)
                cv2.circle(img, (cx, cy), 30, (255, 255, 255), 2)
                cv2.putText(img, "Eraser", (cx, cy-40), 1, 1.5, (255,255,255), 2)

            elif thumb_open: # B. 엄지 펴짐: 쓰기 모드
                if xp == 0 and yp == 0: xp, yp = cx, cy
                
                # 부드러운 선 보정
                nx = int(xp * (1 - SMOOTHING) + cx * SMOOTHING)
                ny = int(yp * (1 - SMOOTHING) + cy * SMOOTHING)

                cv2.line(canvas, (xp, yp), (nx, ny), (0, 255, 0), 10, cv2.LINE_AA)
                cv2.circle(img, (nx, ny), 7, (0, 255, 0), cv2.FILLED)
                xp, yp = nx, ny
                
            else: # C. 엄지 접음: 이동 모드
                xp, yp = 0, 0
                cv2.circle(img, (cx, cy), 10, (0, 0, 255), 2)
                cv2.putText(img, "Moving", (cx, cy-20), 1, 1, (0,0,255), 2)

    # 합성 및 출력
    img_gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    _, img_inv = cv2.threshold(img_gray, 10, 255, cv2.THRESH_BINARY_INV)
    img_inv = cv2.cvtColor(img_inv, cv2.COLOR_GRAY2BGR)
    img = cv2.bitwise_and(img, img_inv)
    img = cv2.bitwise_or(img, canvas)

    cv2.imshow("Thumb Switch Board", img)

    if cv2.waitKey(1) & 0xFF == ord('q'): break
    if cv2.waitKey(1) & 0xFF == ord('c'): canvas = np.zeros_like(img)

cap.release()
cv2.destroyAllWindows()

--- 엄지 스위치 모드 ---
1. 엄지 펼치기: 쓰기 (Green)
2. 엄지 접기: 이동 (Red)
3. 주먹 쥐기: 지우개 (White)


### 파이로매니악(Pyromaniac) AR: 주먹 불꽃 & 장풍

In [64]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, model_complexity=1)
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, img = cap.read()
    if not success: break
    img = cv2.flip(img, -1)
    h, w, _ = img.shape
    
    # 1. 렌더링용 블랙 캔버스 (마야의 렌더 패스 느낌)
    glow_layer = np.zeros_like(img)
    
    result = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            # 손가락 끝점 추출 및 마스크 영역(Hull) 계산
            points = np.array([[int(lm.x * w), int(lm.y * h)] for lm in hand_landmarks.landmark])
            hull = cv2.convexHull(points)
            
            # 주먹 상태 체크
            fingers = [hand_landmarks.landmark[i].y < hand_landmarks.landmark[i-2].y for i in [8, 12, 16, 20]]
            
            if sum(fingers) <= 1: # 주먹 쥐면 '차징'
                # --- 멀티 패스 글로우 합성 ---
                # Pass 1: 코어 (가장 밝은 중심)
                cv2.drawContours(glow_layer, [hull], -1, (0, 150, 255), 1) 
                
                # Pass 2: 미디엄 글로우 (중간 번짐)
                inner_glow = np.zeros_like(img)
                cv2.drawContours(inner_glow, [hull], -1, (0, 50, 255), -1)
                inner_glow = cv2.GaussianBlur(inner_glow, (51, 51), 0)
                
                # Pass 3: 와이드 글로우 (주변 광원)
                outer_glow = np.zeros_like(img)
                cv2.drawContours(outer_glow, [hull], -1, (0, 0, 150), -1)
                outer_glow = cv2.GaussianBlur(outer_glow, (101, 101), 0)
                
                # 패스 합치기 (Linear Dodge 방식)
                glow_layer = cv2.add(glow_layer, inner_glow)
                glow_layer = cv2.add(glow_layer, outer_glow)

    # 2. 최종 합성 (원본 + 글로우 레이어)
    # cv2.add를 쓰면 오버레이 효과(Additive Blending)가 나서 훨씬 화려해짐
    combined = cv2.add(img, glow_layer)

    cv2.imshow("VFX Real-time Compositing", combined)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [68]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, model_complexity=1)
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, img = cap.read()
    if not success: break
    img = cv2.flip(img, -1)
    h, w, _ = img.shape
    
    result = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            points = np.array([[int(lm.x * w), int(lm.y * h)] for lm in hand_landmarks.landmark])
            hull = cv2.convexHull(points)
            
            # 주먹 상태 체크
            fingers = [hand_landmarks.landmark[i].y < hand_landmarks.landmark[i-2].y for i in [8, 12, 16, 20]]
            
            if sum(fingers) <= 1:
                # --- [공식] 정교한 응축 에너지 합성 로직 ---
                
                # 1. 깡통 마스크 생성 (Alpha Mask)
                mask = np.zeros((h, w), dtype=np.uint8)
                cv2.drawContours(mask, [hull], -1, 255, -1) # 꽉 채운 흰색 마스크
                
                # 2. 마스크를 깎아서 '이너 글로우' 만들기 (Erosion)
                # 마야의 'Shrink Selection' 같은 역할
                kernel = np.ones((15, 15), np.uint8)
                inner_mask = cv2.erode(mask, kernel, iterations=1)
                
                # 3. 마스크 소프트니스 (Soft Edge)
                # 외곽선이 튀지 않게 마스크 전체를 가우시안 블러로 뭉갬
                mask_blurred = cv2.GaussianBlur(mask, (51, 51), 0)
                
                # 4. 컬러 레이어 생성 (진한 주황~빨강)
                glow_color = np.zeros_like(img)
                # 안쪽은 밝고 바깥쪽은 어둡게 (Gradient 효과 대용)
                glow_color[:] = (0, 100, 255) # 오렌지색
                
                # 5. 마스크 적용 (Masking)
                # 블러처리된 마스크를 채널로 사용하여 원본에 덧입힘
                glow_final = cv2.bitwise_and(glow_color, glow_color, mask=mask_blurred)
                
                # 6. 최종 합성 (Linear Dodge / Additive)
                img = cv2.add(img, glow_final)

    cv2.imshow("High-End VFX Alpha Blending", img)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

### 화면제어

In [70]:
# !pip install pyautogui

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installi

In [ ]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time

# 설정
pyautogui.FAILSAFE = False
pyautogui.PAUSE = 0
screen_w, screen_h = pyautogui.size()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, model_complexity=0, 
                       min_detection_confidence=0.7, min_tracking_confidence=0.7)

cap = cv2.VideoCapture(0)

# 제어 변수
last_left_time = 0
scroll_mode = False
scroll_start_y = 0

def get_dist(p1, p2):
    return np.hypot(p1.x - p2.x, p1.y - p2.y)

while cap.isOpened():
    success, img = cap.read()
    if not success: break
    
    img = cv2.flip(img, 1)
    h, w, _ = img.shape
    result = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            lm = hand_landmarks.landmark
            
            # 기준점 설정
            pointer_base = lm[5]  # 검지 뿌리 마디 (가장 안정적)
            thumb_tip = lm[4]     # 엄지 끝 (클릭/조합용)
            index_tip = lm[8]     # 검지 끝 (클릭용)
            middle_tip = lm[12]   # 중지 끝 (우클릭용)
            ring_tip = lm[16]     # 약지 끝 (스크롤용)

            # 1. 마우스 이동 (검지 마디 기준)
            # 마디 기준이라 손가락을 까닥여도 커서가 거의 안 흔들림
            mx = np.interp(pointer_base.x, (0.2, 0.8), (0, screen_w))
            my = np.interp(pointer_base.y, (0.2, 0.8), (0, screen_h))
            pyautogui.moveTo(mx, my, _pause=False)

            # 2. 거리 측정
            d_left = get_dist(thumb_tip, index_tip)   # 좌클릭 (엄지-검지)
            d_right = get_dist(thumb_tip, middle_tip) # 우클릭 (엄지-중지)
            d_scroll = get_dist(thumb_tip, ring_tip)  # 스크롤 (엄지-약지)

            # 3. 기능 구현
            # A. 왼쪽 클릭 & 더블클릭
            if d_left < 0.04:
                curr = time.time()
                if curr - last_left_time > 0.2:
                    if curr - last_left_time < 0.5:
                        pyautogui.doubleClick()
                    else:
                        pyautogui.click()
                    last_left_time = curr
                cv2.putText(img, "LEFT/DOUBLE", (50, 50), 1, 2, (0, 255, 0), 2)

            # B. 오른쪽 클릭
            elif d_right < 0.04:
                pyautogui.rightClick()
                time.sleep(0.3)
                cv2.putText(img, "RIGHT", (50, 50), 1, 2, (0, 0, 255), 2)

            # C. 스크롤 모드 (엄지와 약지를 붙이고 손을 위아래로)
            elif d_scroll < 0.04:
                if not scroll_mode:
                    scroll_mode = True
                    scroll_start_y = pointer_base.y
                else:
                    diff = scroll_start_y - pointer_base.y
                    if abs(diff) > 0.01:
                        pyautogui.scroll(int(diff * 1000))
                        # 스크롤 중에는 기준점을 계속 갱신하여 부드럽게 유지
                        scroll_start_y = pointer_base.y 
                cv2.putText(img, "SCROLLING...", (50, 50), 1, 2, (255, 255, 0), 2)
            else:
                scroll_mode = False

    # 화면 표시
    cv2.imshow("Advanced Controller", cv2.resize(img, (640, 480)))
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()